In [1]:
from pynq import Overlay

In [2]:
overlay = Overlay('/home/xilinx/jupyter_notebooks/lab2/design_1.bit')

In [16]:
overlay?

In [4]:
matmul = overlay.matrix_mult_0
matmul?

In [5]:
matmul.register_map

RegisterMap {
  CTRL = Register(AP_START=0, AP_DONE=0, AP_IDLE=1, AP_READY=0, RESERVED_1=0, AUTO_RESTART=0, RESERVED_2=0, INTERRUPT=0, RESERVED_3=0),
  GIER = Register(Enable=0, RESERVED=0),
  IP_IER = Register(CHAN0_INT_EN=0, CHAN1_INT_EN=0, RESERVED_0=0),
  IP_ISR = Register(CHAN0_INT_ST=0, CHAN1_INT_ST=0, RESERVED_0=0)
}

In [6]:
matmul = overlay.matrix_mult_0
bram_a = overlay.axi_bram_ctrl_0
bram_b = overlay.axi_bram_ctrl_1
bram_c = overlay.axi_bram_ctrl_2

In [9]:
N=16
import numpy as np
A = np.random.randint(0, 8, size=(N, N), dtype=np.uint32)
B = np.random.randint(0, 8, size=(N, N), dtype=np.uint32)

In [8]:
def write_mat_to_bram(bram, M):
    for i in range(N):
        for j in range(N):
            idx = i * N + j
            addr = idx * 4          # 32-bit = 4 bytes
            bram.write(addr, int(M[i, j]))
            
def read_mat_from_bram(bram):
    M = np.zeros((N, N), dtype=np.uint32)
    for i in range(N):
        for j in range(N):
            idx = i * N + j
            addr = idx * 4
            M[i, j] = bram.read(addr)
    return M

1) Write A/B to the corresponding BRAM

In [10]:
write_mat_to_bram(bram_a, A)   # a -> bram0
write_mat_to_bram(bram_b, B)   # b -> bram1

2) Start GEMM

In [11]:
matmul.register_map.CTRL.AP_START = 1

3) Polling completed (check AP_IDLE as per experiment requirements)

In [12]:
while matmul.register_map.CTRL.AP_IDLE == 0:
    pass

4) Read back C from bram2

In [13]:
C_hw = read_mat_from_bram(bram_c)   # prod(c) -> bram2

5) Comparison with software gold results

In [14]:
C_sw = (A.astype(np.uint64) @ B.astype(np.uint64)).astype(np.uint32)
ok = np.array_equal(C_hw, C_sw)
print("PASS" if ok else "FAIL")

PASS


---

In [15]:
import time
import numpy as np
from pynq import Overlay

# ---------------------------
# Config
# ---------------------------
BIT_PATH = "/home/xilinx/jupyter_notebooks/lab2/design_1.bit"
N = 16
DTYPE = np.uint32

# ---------------------------
# Load overlay / IP handles
# ---------------------------
overlay = Overlay(BIT_PATH)
matmul = overlay.matrix_mult_0
bram_a = overlay.axi_bram_ctrl_0   # a
bram_b = overlay.axi_bram_ctrl_1   # b
bram_c = overlay.axi_bram_ctrl_2   # prod(c)

# ---------------------------
# Helpers
# ---------------------------
def write_mat_to_bram(bram, M):
    # row-major, 32-bit per element
    for i in range(N):
        for j in range(N):
            idx = i * N + j
            bram.write(idx * 4, int(M[i, j]))

def read_mat_from_bram(bram):
    M = np.zeros((N, N), dtype=DTYPE)
    for i in range(N):
        for j in range(N):
            idx = i * N + j
            M[i, j] = bram.read(idx * 4)
    return M

def gemm_sw(A, B):
    # software-only on PS
    return (A.astype(np.uint64) @ B.astype(np.uint64)).astype(DTYPE)

def run_hw_once(A, B):
    # Load
    write_mat_to_bram(bram_a, A)
    write_mat_to_bram(bram_b, B)

    # Compute trigger
    matmul.register_map.CTRL.AP_START = 1
    while matmul.register_map.CTRL.AP_IDLE == 0:
        pass

    # Unload
    C = read_mat_from_bram(bram_c)
    return C

# ---------------------------
# Test data
# ---------------------------
np.random.seed(7)
A = np.random.randint(0, 16, size=(N, N), dtype=DTYPE)
B = np.random.randint(0, 16, size=(N, N), dtype=DTYPE)

# Warm-up (optional)
_ = run_hw_once(A, B)

# ---------------------------
# Measure software-only
# ---------------------------
t0 = time.time()
C_sw = gemm_sw(A, B)
t1 = time.time()
sw_ms = (t1 - t0) * 1000.0

# ---------------------------
# Measure software+hardware: compute+load/unload
# ---------------------------
t2 = time.time()
C_hw = run_hw_once(A, B)
t3 = time.time()
hw_total_ms = (t3 - t2) * 1000.0

# Optional: measure compute-only HW
# (data already in BRAM from run_hw_once? do a fresh explicit load for fairness)
write_mat_to_bram(bram_a, A)
write_mat_to_bram(bram_b, B)
t4 = time.time()
matmul.register_map.CTRL.AP_START = 1
while matmul.register_map.CTRL.AP_IDLE == 0:
    pass
t5 = time.time()
hw_compute_ms = (t5 - t4) * 1000.0

# ---------------------------
# Correctness check
# ---------------------------
ok = np.array_equal(C_hw, C_sw)

print("=== Timing Results ===")
print(f"Software-only GEMM (PS)                : {sw_ms:.3f} ms")
print(f"Software+Hardware (Load+Compute+Read)  : {hw_total_ms:.3f} ms")
print(f"Hardware Compute-only                  : {hw_compute_ms:.3f} ms")
print(f"Correctness                             : {'PASS' if ok else 'FAIL'}")

if not ok:
    diff = np.where(C_hw != C_sw)
    print("First mismatch index:", (diff[0][0], diff[1][0]))
    i, j = diff[0][0], diff[1][0]
    print("C_hw =", int(C_hw[i, j]), "C_sw =", int(C_sw[i, j]))


=== Timing Results ===
Software-only GEMM (PS)                : 0.914 ms
Software+Hardware (Load+Compute+Read)  : 27.985 ms
Hardware Compute-only                  : 1.243 ms
Correctness                             : PASS
